In [ ]:
"""
Capture matching API responses (e.g. X's SearchTimeline GraphQL calls) from
an already-open, already-logged-in Chrome tab

ONE-TIME SETUP:
1. Quit any running Chrome windows (on Windows, check Task Manager too --
   Chrome sometimes keeps a background process alive).
2. Relaunch Chrome with remote debugging enabled AND origins allowlisted.
   Newer Chrome versions reject WebSocket connections to the debug port
   unless the origin is explicitly allowed -- that's what
   --remote-allow-origins is for (without it you'll get a
   WebSocketBadStatusException: Handshake status 403 Forbidden).

   macOS:
     /Applications/Google\ Chrome.app/Contents/MacOS/Google\ Chrome \
       --remote-debugging-port=9222 --remote-allow-origins=* \
       --user-data-dir="$HOME/chrome-debug-profile"

   Windows (cmd):
     "C:\Program Files\Google\Chrome\Application\chrome.exe" ^
       --remote-debugging-port=9222 --remote-allow-origins=* ^
       --user-data-dir="C:\chrome-debug-profile"

   Linux:
     google-chrome --remote-debugging-port=9222 --remote-allow-origins=* \
       --user-data-dir="$HOME/chrome-debug-profile"

3. Log into x.com and open the search/tab you want to capture from.
"""

import json
import time
import random
import base64
import itertools
import requests
import websocket
from websocket import WebSocketTimeoutException

# ---------------------------------------------------------------- CONFIG --

DEBUGGER_HOST = "127.0.0.1"
DEBUGGER_PORT = 9222
TAB_URL_CONTAINS = "x.com/search"      # which open tab to attach to
URL_FILTERS = ["SearchTimeline"]       # substrings to match in response URLs
OUTPUT_FILE = "captured_responses.jsonl"

AUTO_SCROLL = True                     # nudge the page to trigger more calls
SCROLL_EVERY_SECONDS = 3.0
HEARTBEAT_EVERY_SECONDS = 10.0
RECV_POLL_TIMEOUT = 1.0                # how often the loop wakes up to do housekeeping
MAX_RUNTIME_SECONDS = None             # e.g. 600 to auto-stop after 10 min; None = run until Ctrl+C


def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")


# ------------------------------------------------------------- TAB LOOKUP --

def list_targets():
    resp = requests.get(f"http://{DEBUGGER_HOST}:{DEBUGGER_PORT}/json")
    resp.raise_for_status()
    return resp.json()


def find_target():
    pages = [t for t in list_targets() if t.get("type") == "page"]
    for t in pages:
        if TAB_URL_CONTAINS in t.get("url", ""):
            return t
    log("No tab matched. Currently open tabs:")
    for t in pages:
        log(f"  - {t.get('url')}")
    raise RuntimeError(
        f"No open tab found containing '{TAB_URL_CONTAINS}'. "
        "Open that tab first, or adjust TAB_URL_CONTAINS."
    )


# ----------------------------------------------------------------- CDP I/O --

class CDPClient:

    def __init__(self, ws_url):
        log(f"Connecting to {ws_url}")
        self.ws = websocket.create_connection(ws_url)
        self.ws.settimeout(RECV_POLL_TIMEOUT)
        self._next_id = itertools.count(1)
        self.pending = {}  # command id -> label, filled in by the caller

    def send(self, method, params=None, label=None):
        msg_id = next(self._next_id)
        self.ws.send(json.dumps({"id": msg_id, "method": method, "params": params or {}}))
        if label is not None:
            self.pending[msg_id] = label
        return msg_id

    def poll(self):
        """Return the next parsed message, or None if nothing arrived
        within RECV_POLL_TIMEOUT."""
        try:
            raw = self.ws.recv()
        except WebSocketTimeoutException:
            return None
        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            return None


# ------------------------------------------------------------------- MAIN --

def main():
    target = find_target()
    log(f"Attached to tab: {target['url']}")
    client = CDPClient(target["webSocketDebuggerUrl"])
    client.send("Network.enable")
    log("Network domain enabled. Listening for responses...")

    out = open(OUTPUT_FILE, "a", encoding="utf-8")
    captured = 0
    pending_matches = {}   # requestId -> {"url":, "status":}  (waiting on loadingFinished)
    start_time = time.time()
    last_scroll = 0.0
    last_heartbeat = 0.0

    try:
        while True:
            if MAX_RUNTIME_SECONDS and (time.time() - start_time) > MAX_RUNTIME_SECONDS:
                log(f"Hit MAX_RUNTIME_SECONDS ({MAX_RUNTIME_SECONDS}s), stopping.")
                break

            msg = client.poll()

            if msg is None:
                now = time.time()
                if AUTO_SCROLL and now - last_scroll > SCROLL_EVERY_SECONDS:
                    client.send("Runtime.evaluate", {"expression": "window.scrollBy(0, 2500)"})
                    last_scroll = now
                    log("Scrolled tab to trigger more results.")
                if now - last_heartbeat > HEARTBEAT_EVERY_SECONDS:
                    log(f"Still listening... {captured} captured so far, "
                        f"{len(pending_matches)} response(s) awaiting body.")
                    last_heartbeat = now
                continue

            method = msg.get("method")

            if method == "Network.responseReceived":
                p = msg["params"]
                req_id, resp = p["requestId"], p["response"]
                if any(f in resp["url"] for f in URL_FILTERS):
                    pending_matches[req_id] = {"url": resp["url"], "status": resp["status"]}
                    tag = "WARNING" if resp["status"] >= 400 else "OK"
                    log(f"[{tag}] matched response (status {resp['status']}): {resp['url']}")
                continue

            if method == "Network.loadingFinished":
                req_id = msg["params"]["requestId"]
                if req_id in pending_matches:
                    meta = pending_matches.pop(req_id)
                    client.send(
                        "Network.getResponseBody",
                        {"requestId": req_id},
                        label=("body", meta["url"], meta["status"]),
                    )
                continue

            if method == "Network.loadingFailed":
                req_id = msg["params"]["requestId"]
                if req_id in pending_matches:
                    meta = pending_matches.pop(req_id)
                    log(f"  Request failed before body was available: {meta['url']} "
                        f"({msg['params'].get('errorText')})")
                continue

            # -- command results come back with an "id", not a "method" --
            msg_id = msg.get("id")
            if msg_id is not None and msg_id in client.pending:
                label = client.pending.pop(msg_id)
                if label and label[0] == "body":
                    _, url, status = label
                    result = msg.get("result", {})
                    body_text = result.get("body", "")
                    if result.get("base64Encoded"):
                        try:
                            body_text = base64.b64decode(body_text).decode("utf-8", errors="replace")
                        except Exception as e:
                            log(f"  Could not decode body for {url}: {e}")
                            continue
                    try:
                        body_json = json.loads(body_text)
                    except json.JSONDecodeError:
                        log(f"  Body wasn't valid JSON, skipping: {url}")
                        continue

                    out.write(json.dumps({"url": url, "status": status, "body": body_json}) + "\n")
                    out.flush()
                    captured += 1
                    log(f"  -> Captured #{captured}, saved to {OUTPUT_FILE}")

    except KeyboardInterrupt:
        log("Stopped by user (Ctrl+C).")
    finally:
        out.close()
        log(f"Done. {captured} response bodies written to {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

In [ ]:
import re
from datetime import datetime
import json
import pandas as pd

def get_df(response_json):
    tweets = extract_tweet_attributes(response_json)
    df = pd.DataFrame([
                {
                    'username': tweet['username'],
                    'timestamp': tweet['timestamp'],
                    'content': tweet['content'],
                    'likes': tweet['engagement_metrics']['likes'],
                    'retweets': tweet['engagement_metrics']['retweets'],
                    'replies': tweet['engagement_metrics']['replies'],
                    'quotes': tweet['engagement_metrics']['quotes'],
                    'bookmarks': tweet['engagement_metrics']['bookmarks'],
                    'views': tweet['engagement_metrics']['views'],
                    'mentions': ', '.join(tweet['mentions']) if tweet['mentions'] else '',
                    'hashtags': ', '.join(tweet['hashtags']) if tweet['hashtags'] else '',
                    'tweet_id': tweet['tweet_id'],
                    'url': tweet['url']
                }
                for tweet in tweets
            ])
    return df

def extract_tweet_attributes(response_json):
    """
    Extract tweet attributes from X.com API response.
    Returns a list of dictionaries containing tweet data.
    """
    tweets = []
    
    try:
        entries = response_json['data']['search_by_raw_query']['search_timeline']['timeline']['instructions'][0]['entries']
        
        for entry in entries:
            try:
                # Skip non-tweet entries
                if 'itemContent' not in entry.get('content', {}):
                    continue
                
                tweet_data = entry['content']['itemContent']['tweet_results']['result']
                
                # Handle TweetWithVisibilityResults wrapper (common in some responses)
                if tweet_data.get('__typename') == 'TweetWithVisibilityResults':
                    tweet_data = tweet_data.get('tweet', {})
                
                legacy = tweet_data.get('legacy', {})
                core = tweet_data.get('core', {})
                
                # Views extraction (works for both normal Tweet and unwrapped cases)
                views = tweet_data.get('views', {}).get('count', 'N/A')
                
                # Extract username
                username = core['user_results']['result']['core'].get('screen_name', 'N/A')
                
                # Extract timestamp (created_at format: "Wed Aug 29 12:34:56 +0000 2026")
                timestamp_str = legacy.get('created_at', 'N/A')
                
                # Extract content (full text)
                content = legacy.get('full_text', '')
                
                # Extract engagement metrics
                engagement_metrics = {
                    'likes': legacy.get('favorite_count', 0),
                    'retweets': legacy.get('retweet_count', 0),
                    'replies': legacy.get('reply_count', 0),
                    'quotes': legacy.get('quote_count', 0),
                    'bookmarks': legacy.get('bookmark_count', 0),
                    'views': views
                }
                
                # Extract mentions (using regex on full_text)
                mentions = re.findall(r'@(\w+)', content)
                
                # Extract hashtags (using regex on full_text)
                hashtags = re.findall(r'#(\w+)', content)
                
                tweet_info = {
                    'username': username,
                    'timestamp': timestamp_str,
                    'content': content,
                    'engagement_metrics': engagement_metrics,
                    'mentions': list(set(mentions)),  # Remove duplicates
                    'hashtags': list(set(hashtags)),  # Remove duplicates
                    'tweet_id': legacy.get('id_str', 'N/A'),
                    'url': f"https://x.com/{username}/status/{legacy.get('id_str', '')}" if username else 'N/A'
                }
                
                tweets.append(tweet_info)
                
            except (KeyError, TypeError) as e:
                # Skip entries with missing data
                continue
    
    except (KeyError, IndexError) as e:
        print(f"Error parsing response: {e}")
        return []
    
    return tweets

# code to append new tweets to an existing CSV file without overwriting it
def append_tweets_to_csv(new_tweets_df, csv_file_path):
    """
    Append new tweets to an existing CSV file without overwriting it.
    
    Parameters:
    -----------
    new_tweets_df : pd.DataFrame
        DataFrame containing new tweets to append
    csv_file_path : str
        Path to the existing CSV file
    """
    
    try:
        # Check if the CSV file exists
        try:
            existing_df = pd.read_csv(csv_file_path)
            print(f"Existing CSV found with {len(existing_df)} tweets.")
        except FileNotFoundError:
            existing_df = pd.DataFrame()
            print("No existing CSV found. A new file will be created.")
        
        # Concatenate the new tweets with the existing ones
        combined_df = pd.concat([existing_df, new_tweets_df], ignore_index=True)
        
        # Drop duplicates based on tweet_id to avoid duplicates
        combined_df.drop_duplicates(subset='tweet_id', inplace=True)
        
        # Save back to CSV
        combined_df.to_csv(csv_file_path, index=False)
        print(f"✓ Successfully appended {len(new_tweets_df)} new tweets. Total tweets now: {len(combined_df)}")
    
    except Exception as e:
        print(f"Error while appending tweets to CSV: {e}")



In [ ]:
# read this jsonl file and convert it to a pandas dataframe, then save it as a csv file
with open("captured_responses.jsonl", "r", encoding="utf-8") as f:
    lines = f.readlines()
    for line in lines:
        data = json.loads(line)
        
        df = get_df(data['body'])
        append_tweets_to_csv(df, "tweets.csv")